# Model Inference: Kaggle submission (from the Model Registry)

Loads the champion **strictly from the MLflow Model Registry** (`Walmart_Champion_TimesFM`,
the zero-shot foundation model — Kaggle public 2,969, our best) and runs it on the
**raw** `test.csv` to produce `submission.csv`. All forecasting happens inside the
registered model.

Register the model first with `register_champion_timesfm.ipynb`.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

cwd: D:\Desktop\ML Project


In [2]:
import warnings
import numpy as np
import pandas as pd
import mlflow
import mlflow.pyfunc
import dagshub
from mlflow.tracking import MlflowClient

warnings.filterwarnings("ignore")
dagshub.init(repo_owner="slosa23", repo_name="Walmart-Sales-Forecasting", mlflow=True)
REGISTERED_NAME = "Walmart_Champion_TimesFM"
print("Tracking URI:", mlflow.get_tracking_uri())

D:\Desktop\ML Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Accessing as GigiSichinava

Initialized MLflow to track repo "slosa23/Walmart-Sales-Forecasting"

Repository slosa23/Walmart-Sales-Forecasting initialized!

Tracking URI: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow


## Load the champion from the registry

In [3]:
client = MlflowClient()
versions = client.search_model_versions(f"name='{REGISTERED_NAME}'")
latest = max(int(v.version) for v in versions)
print("Loading", REGISTERED_NAME, "version", latest)
champion = mlflow.pyfunc.load_model(f"models:/{REGISTERED_NAME}/{latest}")

Loading Walmart_Champion_TimesFM version 1


## Predict on the raw test set and write the submission

In [4]:
test_raw = pd.read_csv("data/test.csv")
preds = champion.predict(test_raw[["Store", "Dept", "Date", "IsHoliday"]])

submission = pd.DataFrame({
    "Id": test_raw["Store"].astype(str) + "_" + test_raw["Dept"].astype(str) + "_" + test_raw["Date"].astype(str),
    "Weekly_Sales": np.asarray(preds, dtype=float),
})
submission.to_csv("submission.csv", index=False)

print("rows:", len(submission), "| NaNs:", int(submission["Weekly_Sales"].isna().sum()),
      "| negatives:", int((submission["Weekly_Sales"] < 0).sum()))
submission.head()

rows: 115064 | NaNs: 0 | negatives: 0


,Id,Weekly_Sales
0,1_1_2012-11-02,29630.828125
1,1_1_2012-11-09,24433.855469
2,1_1_2012-11-16,21384.291016
3,1_1_2012-11-23,21034.988281
4,1_1_2012-11-30,25151.812500


## Notes

- The champion is loaded from the Model Registry (not a local run) and runs on raw
  `test.csv`, as required.
- TimesFM (zero-shot) scored public 2,969 / private 3,130 on Kaggle, our best, ahead of
  the ensemble (3,923) and N-BEATS (4,143).